<a href="https://www.kaggle.com/code/rishabhbhartiya/credit-notebook?scriptVersionId=374823539" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Playground S6E11 - Credit Card Fraud (final pipeline)
Metric: ROC AUC. Run with **Save Version -> Save & Run All (Commit)**, then submit `submission.csv` from the Output tab.

**Findings behind this pipeline**
- The label is mostly *additive*: XGB depth-1 (0.9467) ~ depth-2 (0.9471) > depth-6 (0.9459)
- No `id` leak, adversarial AUC = 0.50 (train/test share one distribution), the original dataset gave no gain
- So: shallow XGBoost (GPU) + native-categorical CatBoost, seed-averaged, rank-blended

In [1]:
# %% 0. Config
import os

MODE   = "blend"            # "blend" = blend of all models | "safe" = xgb_d2_pair only (seed-averaged)
SEEDS  = [42, 7, 2026]      # use [42] for a faster run
NFOLD  = 5
DEVICE = "cuda"             # needs GPU (Settings -> Accelerator)
DIR    = "/kaggle/input/competitions/playground-series-s6e11"
OUT    = "/kaggle/working"

In [2]:
# %% 1. Imports + data
import warnings
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")

TARGET, ID = "is_fraud", "id"
train = pd.read_csv(f"{DIR}/train.csv")
test  = pd.read_csv(f"{DIR}/test.csv")
sub   = pd.read_csv(f"{DIR}/sample_submission.csv")
y = train[TARGET].values
n_tr, n_te = len(train), len(test)
print(train.shape, test.shape, "fraud rate:", y.mean())

(699960, 26) (299984, 25) fraud rate: 0.016699525687182123


In [3]:
# %% 2. Feature engineering (train+test together, unsupervised, no target used)
raw = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True).drop(columns=[ID])
bools    = raw.select_dtypes("bool").columns.tolist()
raw[bools] = raw[bools].astype(int)
cat_cols = raw.select_dtypes("object").columns.tolist()

def fe(d):
    d = d.copy()
    d["log_amt"]        = np.log1p(d.amount_usd)
    d["amt_bal_ratio"]  = d.amount_usd / d.account_balance_usd
    d["amt_per_txn"]    = d.amount_usd / (d.txn_count_last_24h + 1)
    d["dist_per_hour"]  = d.distance_from_home_km / (d.hours_since_last_txn + 0.1)
    d["vel_x_cnt"]      = d.velocity_score * d.txn_count_last_24h
    d["vel_x_merch"]    = d.velocity_score * d.merchant_risk_score
    d["amt_x_merch"]    = d.log_amt * d.merchant_risk_score
    d["night"]          = d.time_of_day_hour.isin([0, 1, 2, 3, 4, 5]).astype(int)
    d["hour_sin"]       = np.sin(2 * np.pi * d.time_of_day_hour / 24)
    d["hour_cos"]       = np.cos(2 * np.pi * d.time_of_day_hour / 24)
    d["risk_flags"]     = d[bools].sum(axis=1)
    d["cvv_disp"]       = d.cvv_retry_count + d.prior_disputes
    for g in ["merchant_category", "channel", "card_type"]:
        d[f"amt_vs_{g}"] = d.amount_usd / d.groupby(g).amount_usd.transform("mean")
    for c in cat_cols:
        d[c] = d[c].astype("category").cat.codes
    return d

F = fe(raw)
X,  X_test  = F.iloc[:n_tr].reset_index(drop=True), F.iloc[n_tr:].reset_index(drop=True)

# pair features (cvv x auth x flags) -> used by xgb_d2_pair
X_pair, Xt_pair = X.copy(), X_test.copy()
for A in (X_pair, Xt_pair):
    A["cvv_auth"]   = A.cvv_retry_count * 10 + A.auth_method
    A["flags_cvv"]  = A.risk_flags * 10 + A.cvv_retry_count
    A["flags_auth"] = A.risk_flags * 10 + A.auth_method

# int-coded categoricals for CatBoost's native categorical handling
X_cat, Xt_cat = X.copy(), X_test.copy()
for A in (X_cat, Xt_cat):
    for c in cat_cols:
        A[c] = A[c].astype(int)
print("features:", X.shape[1], "| with pairs:", X_pair.shape[1])

features: 39 | with pairs: 42


In [4]:
# %% 3. CV runner (seed-averaged) + model factories
def run_cv(name, fit_predict, Xd, Xtd, seeds=SEEDS):
    oof, pred = np.zeros(n_tr), np.zeros(n_te)
    for s in seeds:
        skf = StratifiedKFold(NFOLD, shuffle=True, random_state=s)
        o = np.zeros(n_tr)
        for tr, va in skf.split(Xd, y):
            p_va, p_te = fit_predict(Xd.iloc[tr], y[tr], Xd.iloc[va], y[va], Xtd, s)
            o[va] = p_va
            pred += p_te / (NFOLD * len(seeds))
        oof += o / len(seeds)
        print(f"[{name}] seed {s}: OOF AUC = {roc_auc_score(y, o):.5f}")
    print(f"[{name}] >>> averaged OOF AUC = {roc_auc_score(y, oof):.5f}\n")
    return oof, pred

def make_xgb(depth, lr=0.06, mcw=5):
    def fp(Xtr, ytr, Xva, yva, Xte, seed):
        m = xgb.XGBClassifier(
            n_estimators=6000, learning_rate=lr, max_depth=depth, min_child_weight=mcw,
            subsample=0.8, colsample_bytree=0.7, reg_lambda=5.0,
            tree_method="hist", device=DEVICE, eval_metric="auc",
            early_stopping_rounds=200, random_state=seed)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    return fp

def fit_cat(Xtr, ytr, Xva, yva, Xte, seed):
    m = CatBoostClassifier(iterations=4000, learning_rate=0.06, depth=4, l2_leaf_reg=10,
                           eval_metric="AUC", random_seed=seed, early_stopping_rounds=200,
                           cat_features=cat_cols, verbose=0, thread_count=-1)
    m.fit(Xtr, ytr, eval_set=(Xva, yva))
    return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]

In [5]:
# %% 4. Train models
results = {}
results["xgb_d2_pair"] = run_cv("xgb_d2_pair", make_xgb(2, lr=0.03, mcw=10), X_pair, Xt_pair)

if MODE == "blend":
    results["xgb_d1"]     = run_cv("xgb_d1",     make_xgb(1, lr=0.08, mcw=5),  X, X_test)
    results["xgb_d2"]     = run_cv("xgb_d2",     make_xgb(2, lr=0.06, mcw=5),  X, X_test)
    results["xgb_d2_reg"] = run_cv("xgb_d2_reg", make_xgb(2, lr=0.02, mcw=30), X, X_test)
    results["xgb_d3"]     = run_cv("xgb_d3",     make_xgb(3, lr=0.06, mcw=5),  X, X_test)
    results["cat_native"] = run_cv("cat_native", fit_cat, X_cat, Xt_cat, seeds=SEEDS[:1])

[xgb_d2_pair] seed 42: OOF AUC = 0.94714
[xgb_d2_pair] seed 7: OOF AUC = 0.94715
[xgb_d2_pair] seed 2026: OOF AUC = 0.94728
[xgb_d2_pair] >>> averaged OOF AUC = 0.94742

[xgb_d1] seed 42: OOF AUC = 0.94670
[xgb_d1] seed 7: OOF AUC = 0.94673
[xgb_d1] seed 2026: OOF AUC = 0.94678
[xgb_d1] >>> averaged OOF AUC = 0.94689

[xgb_d2] seed 42: OOF AUC = 0.94698
[xgb_d2] seed 7: OOF AUC = 0.94715
[xgb_d2] seed 2026: OOF AUC = 0.94716
[xgb_d2] >>> averaged OOF AUC = 0.94742

[xgb_d2_reg] seed 42: OOF AUC = 0.94709
[xgb_d2_reg] seed 7: OOF AUC = 0.94711
[xgb_d2_reg] seed 2026: OOF AUC = 0.94726
[xgb_d2_reg] >>> averaged OOF AUC = 0.94735

[xgb_d3] seed 42: OOF AUC = 0.94686
[xgb_d3] seed 7: OOF AUC = 0.94680
[xgb_d3] seed 2026: OOF AUC = 0.94699
[xgb_d3] >>> averaged OOF AUC = 0.94735

[cat_native] seed 42: OOF AUC = 0.94613
[cat_native] >>> averaged OOF AUC = 0.94613



In [6]:
# %% 5. Blend -> submission.csv (rank average, weights optimised on OOF AUC)
names = list(results)
oofs  = np.column_stack([rankdata(results[n][0]) / n_tr for n in names])
tests = np.column_stack([rankdata(results[n][1]) / n_te for n in names])

if len(names) == 1:
    w = np.array([1.0])
    blend_auc = roc_auc_score(y, oofs[:, 0])
else:
    neg = lambda w: -roc_auc_score(y, oofs @ (np.abs(w) / np.abs(w).sum()))
    r = minimize(neg, np.ones(len(names)) / len(names), method="Nelder-Mead", options={"maxiter": 2000})
    w = np.abs(r.x) / np.abs(r.x).sum()
    blend_auc = -r.fun
print({n: round(float(x), 3) for n, x in zip(names, w)}, "| blend OOF AUC:", round(blend_auc, 5))

sub[TARGET] = tests @ w
sub.to_csv(f"{OUT}/submission.csv", index=False)

# sanity checks
chk = pd.read_csv(f"{OUT}/submission.csv")
assert list(chk.columns) == [ID, TARGET]
assert len(chk) == n_te and chk[ID].equals(test[ID]), "id mismatch"
assert chk[TARGET].notna().all() and chk[TARGET].between(0, 1).all()
print(chk.head(), "\nfiles in output dir:", os.listdir(OUT))

{'xgb_d2_pair': 0.337, 'xgb_d1': 0.049, 'xgb_d2': 0.227, 'xgb_d2_reg': 0.002, 'xgb_d3': 0.324, 'cat_native': 0.061} | blend OOF AUC: 0.94752
       id  is_fraud
0  699960  0.152869
1  699961  0.067568
2  699962  0.948378
3  699963  0.447572
4  699964  0.380365 
files in output dir: ['__notebook__.ipynb', 'catboost_info', 'submission.csv']
